# Bangladesh Legal Assistant — Cloud SFT (Qwen2.5-7B-Instruct + QLoRA)

Self-contained Colab notebook. Runs on a **free T4 (16 GB)** with 4-bit quantization, or much faster on an **A100 / L4 / A10G**.

Inputs:
- A **private HF dataset repo** (produced by `push_to_hub.py`) with `train` and `validation` splits.
- An HF token with `write` permission (for pushing the trained adapter back).

Outputs:
- A LoRA adapter pushed to `HF_OUTPUT_REPO` on the Hub.
- An eval report saved alongside the adapter.

## 1. Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
    "peft>=0.12.0" "bitsandbytes>=0.43.0" "trl>=0.10.1" "huggingface_hub>=0.24.0" \
    "sentencepiece>=0.2.0" "protobuf>=4.25.0" "einops>=0.8.0"

## 2. Authenticate with the Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login
from getpass import getpass

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste your HF token (write scope): ')
login(token=os.environ['HF_TOKEN'])
print('logged in')

## 3. Configuration (edit me)

In [ ]:
# Base model and target output repos.
BASE_MODEL       = 'Qwen/Qwen2.5-7B-Instruct'
DATA_REPO        = 'tanziro/bd-legal-sft'        # private HF dataset id from push_to_hub.py
HF_OUTPUT_REPO   = 'tanziro/bd-legal-qwen25-7b-lora'  # adapter destination

# Training hyperparameters tuned for a free T4 with QLoRA.
MAX_LEN          = 1024
BATCH_SIZE       = 1
GRAD_ACCUM       = 16
LEARNING_RATE    = 2e-4
EPOCHS           = 1.0
WARMUP_RATIO     = 0.03
LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
SEED             = 42
OUTPUT_DIR       = '/content/legal-sft-out'

## 4. Load the SFT dataset from the Hub

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATA_REPO, token=os.environ['HF_TOKEN'])
print(ds)
print('train sample:', ds['train'][0])

## 5. Prompt rendering + tokenization

Mirrors `train.py` so cloud + local training produce identical chat templates. Loss is computed only on the `<RESPONSE>` tokens.

In [ ]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    'You are a Bangladesh legal research assistant. You are not a lawyer. '
    'Cite the official Laws of Bangladesh portal. '
    'Refuse if the retrieved evidence is insufficient.'
)

def render_prompt(row):
    return (
        f"<SYSTEM>{SYSTEM_PROMPT}</SYSTEM>\n"
        f"<INSTRUCTION>{row.get('instruction','')}</INSTRUCTION>\n"
        f"<CONTEXT>{row.get('context','')}</CONTEXT>\n"
        f"<RESPONSE>"
    )

def render_full(row):
    return render_prompt(row) + f"{row.get('response','')}</RESPONSE>"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(row):
    prompt = render_prompt(row)
    full   = render_full(row)
    enc_full   = tokenizer(full, truncation=True, max_length=MAX_LEN, padding='max_length')
    enc_prompt = tokenizer(prompt, truncation=True, max_length=MAX_LEN, add_special_tokens=False)
    labels = list(enc_full['input_ids'])
    plen = min(len(enc_prompt['input_ids']), len(labels))
    for i in range(plen):
        labels[i] = -100
    pad = tokenizer.pad_token_id
    for i, t in enumerate(enc_full['input_ids']):
        if t == pad:
            labels[i] = -100
    enc_full['labels'] = labels
    return enc_full

ds_tok = ds.map(tokenize, remove_columns=ds['train'].column_names, desc='tokenize', num_proc=2)
print(ds_tok)

## 6. Load Qwen2.5-7B-Instruct in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.0,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    report_to=[],
    push_to_hub=False,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 8. Quick eval — perplexity + citation-presence rate on validation

In [ ]:
import math, json

model.eval()
rows = list(ds['validation'])[:64]
total_loss = 0.0; total_tok = 0
with torch.no_grad():
    for r in rows:
        enc = tokenizer(render_full(r), return_tensors='pt', truncation=True, max_length=MAX_LEN).to(model.device)
        out = model(**enc, labels=enc['input_ids'])
        n = enc['input_ids'].numel()
        total_loss += out.loss.item() * n
        total_tok += n
ppl = math.exp(min(total_loss/total_tok, 20))
print('perplexity_first64:', round(ppl, 3))

hits = total = 0
samples = []
for r in rows[:8]:
    enc = tokenizer(render_prompt(r), return_tensors='pt', truncation=True, max_length=MAX_LEN).to(model.device)
    gen = model.generate(**enc, max_new_tokens=160, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    txt = tokenizer.decode(gen[0], skip_special_tokens=True)
    if r['task_type'] != 'refusal' and r.get('source_url'):
        total += 1
        if r['source_url'] in txt:
            hits += 1
    samples.append({'task_type': r['task_type'], 'instruction': r['instruction'][:200],
                    'generated': txt[-400:], 'source_url': r.get('source_url','')})
report = {
    'base_model': BASE_MODEL,
    'data_repo': DATA_REPO,
    'perplexity_first64': ppl,
    'citation_presence_rate': (hits/total) if total else 0.0,
    'samples': samples,
}
with open(f'{OUTPUT_DIR}/eval_report.json', 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print('citation_presence_rate:', report['citation_presence_rate'])

## 9. Push the adapter (and eval report) back to the Hub

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(repo_id=HF_OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
upload_folder(
    repo_id=HF_OUTPUT_REPO,
    folder_path=OUTPUT_DIR,
    repo_type='model',
    commit_message='SFT LoRA adapter for Bangladesh legal assistant',
)
print('adapter pushed:', f'https://huggingface.co/{HF_OUTPUT_REPO}')

## 10. (Optional) Inference sanity check

Reload the adapter from the Hub to confirm it loads cleanly. Use a low temperature — this is a research assistant, not a brainstorm partner.

In [ ]:
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
infer_model = PeftModel.from_pretrained(base, HF_OUTPUT_REPO)
infer_model.eval()

test = {
    'instruction': 'What does section 302 of the Penal Code provide? Quote the operative text and cite the source.',
    'context': '',
}
enc = tokenizer(render_prompt(test), return_tensors='pt').to(infer_model.device)
out = infer_model.generate(**enc, max_new_tokens=220, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))